# 📝 평가셋 구축 과제 정답 — 내 손으로 만들고, 그 평가셋으로 재기 (강사용)

각 문제의 **모범답안 + 해설**입니다. 경로는 `../../day20_ReAct_멀티툴_에이전트/data/` 입니다.

- 이 과제는 **모델을 한 번도 부르지 않습니다** — 임베딩과 검색만 씁니다.
- **4번부터는 학생마다 평가셋이 다릅니다.** 그래서 채점이 값을 고정하지 않고 **어떤 평가셋에서도 참인 성질**만 봅니다 — 정답 id 가 코퍼스에 있는가, 근거가 원문에 있는가, K 를 키우면 Hit·Recall·MRR 이 줄지 않는가. 아래 모범답안의 수치는 **이 8문항 기준**이며 학생 값과 다른 것이 정상입니다.
- 4번 모범답안의 질문 8개는 **일부러 겹치는 항목**(연차~촉진, 경조휴가~경조사비, 재택~근태)을 노려 복수 정답 문항을 만들었습니다. 학생이 다른 조합을 골라도 조건만 지켰으면 맞습니다.

## 1. 재는 대상 살펴보기

**배경**: 평가셋을 만들기 전에 **무엇을 재는지**부터 알아야 합니다. 어떤 주제가 몇 건씩 들어 있는지, 한 건이 얼마나 긴지 보지 않고 질문부터 쓰면, 문서에 없는 것을 묻거나 한 주제만 잔뜩 묻게 됩니다.

**요구사항**:
- `data/hr_faq.csv` 를 읽어 변수 **`faq`** 에 담으세요.
- 문서 수를 `문서 24건` 형태로 출력하세요.
- `display()` 로 **앞 3행**을 보세요.
- `분류` 열의 값별 개수를 출력하세요.
- 본문 평균 글자 수를 **정수로** 출력하세요.

**예시**

```
문서 24건
(앞 3행 표)
분류
휴가      8
...
본문 평균 글자 수: 101
```

<details><summary>힌트</summary>

```text
접근방법:
- 읽고, 세고, 눈으로 본다. 새로운 것은 없다.

세부구현:
1. pandas 로 CSV 를 읽는다.
2. 행 수는 len 으로 센다.
3. 분류별 개수는 값 세기 메서드로, 표로 찍히니 문자열로 바꿔 출력한다.
4. 본문 길이는 문자열 길이 accessor 로 재고 평균을 낸 뒤 정수로 바꾼다.
```

</details>

In [ ]:
import pandas as pd

faq = pd.read_csv('../../day20_ReAct_멀티툴_에이전트/data/hr_faq.csv')
print(f'문서 {len(faq)}건')

# 한 행이 한 문서다 -- 본문까지 표에 넣으면 잘려 보이므로 앞 3행만 눈으로 확인한다
display(faq.head(3))

# 한 주제만 잔뜩 들어 있으면 질문도 그쪽으로 쏠린다
print(faq['분류'].value_counts().to_string())
print('본문 평균 글자 수:', int(faq['본문'].str.len().mean()))

In [ ]:
# [자가채점]
assert len(faq) == 24, 'hr_faq.csv 를 읽어 faq 에 담으세요'
assert list(faq.columns) == ['id', '분류', '제목', '본문'], '열 이름이 다릅니다'
assert faq['id'].is_unique, '문서 id 는 유일해야 합니다'
print('✅ 통과!')

**해설**: 이 코퍼스는 한 건이 100자 남짓이라 **자를 것이 없습니다.** 안내서처럼 긴 문서를 다룰 때는 먼저 조각으로 자르고 그 조각에 라벨을 붙여야 하지만, 여기서는 **한 행이 곧 한 문서**라 라벨이 FAQ 의 id 그대로입니다. 평가셋을 처음 만들어 볼 때 이 구조가 편한 이유입니다 — **자르는 규칙이 바뀌어 라벨이 어긋나는 문제**가 아예 없습니다.

분류 분포(휴가 8·복리후생 6·근무 5·급여 4·증명 1)를 먼저 본 이유도 있습니다. 4번에서 질문을 쓸 때 휴가만 여덟 개 묻는 평가셋이 되면, 그 평가셋은 **휴가 문서를 잘 찾는지**만 재게 됩니다.

## 2. 색인 만들기 — 한 행이 한 문서

**배경**: 검색을 재려면 검색기가 있어야 합니다. 자를 것이 없으니 **한 행을 그대로 문서 하나로** 감싸 색인합니다. 평가에서 필요한 것은 검색 결과의 본문이 아니라 **id** 이므로(정답 라벨이 id 니까요) id 만 꺼내 주는 함수를 만들어 둡니다.

**요구사항**:
- 각 행을 `Document` 로 감싸 리스트 **`documents`** 를 만드세요.
  - `page_content` 는 **제목과 본문을 줄바꿈(`\n`)으로 이은 문자열**입니다(제목도 검색에 도움이 됩니다).
  - `metadata` 는 **`{'doc_id': 그 행의 id}`** 입니다.
- 임베딩은 **`jhgan/ko-sroberta-multitask`** 를 쓰고, `Chroma.from_documents` 로 색인 **`store`** 를 만드세요. `collection_name` 은 **`'hr_faq'`**, `ids=` 에는 FAQ 의 **id 목록을 그대로** 넘깁니다.
- 함수 **`search_ids(query, k) -> list`** 를 정의하세요. 질문과 가장 가까운 문서 `k`개의 **`doc_id` 를 1위부터 순서대로** 담은 리스트를 돌려줍니다.

**예시**

```
search_ids('연말정산은 언제 하나요?', 3)   -> ['h17', 'h16', 'h23']   (사람마다 뒤 순위는 다를 수 있습니다)
```

> 색인을 만들 때 임베딩 모델을 내려받느라 **처음 한 번은 잠시 걸립니다.** 이 셀은 한 번만 실행하고 끝까지 재사용하세요.

<details><summary>힌트</summary>

```text
접근방법:
- 자를 것이 없으니 반복문 한 번으로 문서 리스트가 끝난다.
- 검색기는 색인을 감싸 만들고, 결과에서 꼬리표만 꺼낸다.

세부구현:
1. 행을 돌며 제목과 본문을 줄바꿈으로 이어 page_content 로, id 를 꼬리표로 넣어 Document 를 만든다.
2. 임베딩 객체를 만들고 색인에 문서 리스트를 넣는다. 이때 id 목록도 함께 넘긴다.
3. 함수 안에서 색인을 검색기로 바꾸되 상위 몇 개를 볼지는 인자로 받은 값을 쓴다.
4. 검색 결과 하나하나에서 꼬리표의 doc_id 만 꺼내 리스트로 돌려준다.
```

</details>

In [ ]:
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

# 한 행이 한 문서라 자르는 단계가 아예 없다 -- 제목을 앞에 붙여 검색이 주제를 함께 보게 한다
documents = [Document(page_content=f'{row.제목}\n{row.본문}', metadata={'doc_id': row.id})
             for row in faq.itertuples()]

embeddings = HuggingFaceEmbeddings(model_name='jhgan/ko-sroberta-multitask')

# ids= 로 우리 id 를 그대로 넘기면 같은 id 는 덮어쓰기가 된다 -> 이 셀을 다시 실행해도 중복되지 않는다
store = Chroma.from_documents(documents, embeddings,
                              collection_name='hr_faq', ids=list(faq['id']))


def search_ids(query, k):
    """질문과 가장 가까운 문서 k개의 doc_id 를 순위 순서로 돌려준다."""
    # k 를 바꿔 가며 재야 하므로 검색기는 부를 때마다 여기서 만든다(색인을 감싸기만 해 부담이 없다)
    retriever = store.as_retriever(search_kwargs={'k': k})
    return [d.metadata['doc_id'] for d in retriever.invoke(query)]


print(search_ids('연말정산은 언제 하나요?', 3))

In [ ]:
# [자가채점]
found = search_ids('재택근무는 며칠까지 신청할 수 있나요?', 3)
assert isinstance(found, list) and len(found) == 3, '문서 id 3개를 담은 리스트를 돌려주세요'
assert all(f in set(faq['id']) for f in found), 'doc_id 만 담아야 합니다(본문이 아니라)'
assert len(set(found)) == 3, '같은 문서가 두 번 들어왔습니다 - ids= 를 넘겼는지 확인하세요'
assert 'h10' in found, '재택근무 문서가 상위 3개에 없습니다 - page_content 를 확인하세요'
print('✅ 통과!')

**해설**: 눈여겨볼 곳은 셋입니다.

1. **`metadata` 에 `doc_id` 를 넣는 것** — 평가에서 필요한 것은 본문이 아니라 id 입니다. 여기서 빠뜨리면 검색은 되는데 **잰 결과를 정답과 맞춰 볼 수가 없습니다.**
2. **`ids=` 를 함께 넘기는 것** — 같은 id 는 덮어쓰기가 되므로 셀을 여러 번 실행해도 문서가 중복되지 않습니다. 넘기지 않으면 두 번 실행했을 때 같은 문서가 두 벌 들어가 상위 결과에 같은 내용이 두 번 나옵니다(자가채점의 `len(set(found)) == 3` 이 그것을 봅니다).
3. **제목을 `page_content` 앞에 붙인 것** — 본문에 '재택'이라는 말이 한 번밖에 안 나와도 제목이 '재택근무'면 그 문서가 위로 올라옵니다. 짧은 FAQ 에서는 제목이 본문만큼 값이 있습니다.

**흔한 실수**: `search_ids` 안에서 `k` 를 쓰지 않고 검색기를 노트북 위쪽에서 한 번만 만들어 두면, 6번에서 K 를 바꿔 가며 재는 순간 **K 와 상관없이 같은 개수**가 돌아옵니다. 에러는 나지 않고 표만 조용히 이상해집니다.

## 3. 근거 문장을 검증하는 도구 만들기

**배경**: 라벨에 **근거 문장**을 함께 적어 두면 나중에 사람이 검수할 수 있고, 색인을 다시 만들어도 그 문장을 찾아 라벨을 다시 붙일 수 있습니다. 그런데 사람이 옮겨 적다 보면 원문과 조금씩 달라집니다. **줄바꿈·띄어쓰기 차이는 넘어가되, 원문에 없는 문장은 걸러야** 합니다. 4번에서 여러분이 적을 근거를 5번에서 이 도구로 검사할 것입니다.

**요구사항**: 두 함수와 사전 하나를 만드세요.
- **`body`** — 문서 id 로 본문을 바로 꺼낼 수 있는 사전(`{'h01': '입사 첫해에는 ...', ...}`).
- **`squeeze(text) -> str`** — 공백(띄어쓰기·줄바꿈)을 **모두** 지운 문자열을 돌려줍니다.
- **`quoted_in_faq(sentence, doc_id) -> bool`** — 그 문장이 `doc_id` 문서의 **본문 안에 실제로 있으면** `True`, 아니면 `False`. 공백 차이는 무시합니다.
  - **빈 문자열이나 공백뿐인 문장은 `False`** 입니다. 공백을 지우면 빈 문자열이 되는데, 빈 문자열은 어떤 본문에도 '들어 있다'가 되어 버리기 때문입니다.

**예시**

```
squeeze(' 연차는  하루\n단위 ')                              -> '연차는하루단위'
quoted_in_faq('남은 연차는 다음 해로 이월되지 않습니다', 'h01')   -> True
quoted_in_faq('남은 연차는 수당으로 지급됩니다', 'h01')          -> False   (원문에 없는 문장)
quoted_in_faq('', 'h01')                                  -> False
```

<details><summary>힌트</summary>

```text
접근방법:
- 공백만 지우고 나면 '들어 있나'는 부분문자열 검사 한 줄이다.
- 빈 문자열을 먼저 걸러야 한다. 안 그러면 무엇이든 통과한다.

세부구현:
1. id 와 본문을 짝지어 사전으로 만든다.
2. 정규식으로 공백 문자를 전부 빈 문자열로 바꾸는 함수를 만든다.
3. 검증 함수는 먼저 문장의 앞뒤 공백을 떼고 비어 있으면 바로 거짓을 돌려준다.
4. 비어 있지 않으면 양쪽 모두 공백을 지운 뒤 포함 여부를 돌려준다.
```

</details>

In [ ]:
import re

# id 로 본문을 바로 꺼내 쓰려고 사전으로 만들어 둔다(라벨을 눈으로 검수할 때도 쓴다)
body = dict(zip(faq['id'], faq['본문']))


def squeeze(text):
    """공백(띄어쓰기·줄바꿈)을 모두 지운다 -- 표기 차이로 검증이 실패하지 않게."""
    return re.sub(r'\s+', '', text)


def quoted_in_faq(sentence, doc_id):
    """인용한 문장이 그 문서 본문 안에 실제로 있는지 확인한다."""
    if not sentence.strip():          # 빈 문자열은 어떤 본문에도 '들어 있다' 가 되어 버린다
        return False
    return squeeze(sentence) in squeeze(body[doc_id])


print(quoted_in_faq('남은 연차는 다음 해로 이월되지 않습니다', 'h01'))
print(quoted_in_faq('남은 연차는 수당으로 지급됩니다', 'h01'))

In [ ]:
# [자가채점]
assert squeeze(' 연차는  하루\n단위 ') == '연차는하루단위'
assert quoted_in_faq('남은 연차는 다음 해로 이월되지 않습니다', 'h01') is True
# 줄바꿈·띄어쓰기만 다른 인용은 통과시켜야 한다
assert quoted_in_faq('남은 연차는  다음 해로\n이월되지 않습니다', 'h01') is True
assert quoted_in_faq('남은 연차는 수당으로 지급됩니다', 'h01') is False, '원문에 없는 문장입니다'
assert quoted_in_faq('', 'h01') is False, '빈 문자열은 False 여야 합니다'
assert quoted_in_faq('   ', 'h01') is False, '공백뿐인 문장도 False 여야 합니다'
assert len(body) == 24 and body['h01'].startswith('입사 첫해'), 'body 는 id 로 본문을 꺼내는 사전입니다'
print('✅ 통과!')

**해설**: 이 검사가 잡아 주는 것은 **지어낸 근거**입니다. 나중에 라벨링을 모델에게 시키면 모델이 그럴듯한 문장을 만들어 근거라고 내미는 일이 실제로 생기는데, 이 한 줄이 그것을 거릅니다. 사람이 손으로 적을 때도 마찬가지입니다 — 기억으로 요약해 적으면 원문과 달라집니다.

**빈 문자열 검사를 먼저 하는 이유**: 파이썬에서 `'' in '아무 글자'` 는 `True` 입니다. 그래서 근거를 안 적은 문항이 **전부 통과**해 버립니다. 검증 도구가 아무것도 검증하지 않게 되는 가장 조용한 방식이라, 이 한 줄이 도구 전체의 값어치를 지킵니다.

**공백을 지우고 비교하는 이유**: CSV 에서 옮겨 적거나 줄을 바꿔 적으면 글자는 같아도 문자열은 달라집니다. 여기서 걸리면 **맞는 라벨이 틀린 것으로** 버려집니다. 반대로 공백을 지운다고 지어낸 문장이 통과되지는 않으므로, 이 완화는 안전합니다.

## 4. 평가셋 만들기 — 질문 8개에 라벨 붙이기

**배경**: 이제 **재는 도구**를 만듭니다. 평가셋 한 문항은 세 가지입니다 — **질문**, 그 답이 들어 있는 **문서 id**, 그리고 그 문서에서 답이 되는 **문장**. 앞의 둘만 있어도 점수는 나오지만, 근거 문장이 없으면 나중에 그 라벨이 맞는지 아무도 확인할 수 없습니다.

**요구사항**: FAQ 24건을 직접 읽고 질문 **8개 이상**을 써서 DataFrame **`my_eval`** 에 담으세요.

- 열 이름과 순서는 **`query_id`, `query`, `gold_ids`, `evidence`** 입니다.
  - `query_id` — `q1` 부터 순서대로
  - `query` — 사내 직원이 실제로 물어볼 법한 말투의 질문
  - `gold_ids` — 정답 문서 id 를 **`'|'`** 로 이어 붙인 문자열 (예: `'h04|h05'`)
  - `evidence` — 근거 문장을 **`' || '`**(공백-파이프-파이프-공백)로 이어 붙인 문자열. **`gold_ids` 와 같은 순서, 같은 개수**여야 합니다.
- 근거 문장은 **본문에서 그대로** 떼어 오세요(요약하거나 고쳐 쓰면 5번 검증에서 걸립니다).
- **원문 문장을 그대로 복사해 질문으로 만들지 마세요.** 그러면 검색이 너무 쉽게 맞혀 점수가 실제보다 높게 나옵니다(5번에서 이것도 잽니다).

**세 가지 조건**을 지켜야 합니다.

1. **정답이 2개 이상인 문항이 2개 이상** — 답이 한 문서에만 있는 질문만 모으면 Recall 이 Hit 과 늘 같은 값이 되어, 지표를 넷 만들어 놓고 사실은 둘만 보게 됩니다.
2. **정답이 4개 이상인 '넓은 질문'이 정확히 1개** — 여러 문서에 답이 흩어진 질문입니다. 6번에서 이 문항이 무엇을 망가뜨리는지 보고 **직접 고칠** 것입니다.
3. **같은 질문을 두 번 넣지 않기**

**예시** (형식만 보여 주는 것입니다 — 질문은 직접 쓰세요)

```
query_id  query                          gold_ids   evidence
q1        (질문)                          h04|h05    (h04 의 문장) || (h05 의 문장)
q2        (질문)                          h15        (h15 의 문장)
```

<details><summary>힌트</summary>

```text
접근방법:
- 먼저 표를 눈으로 훑으며 '이건 물어볼 만하다' 싶은 것을 고른다.
- 답이 두 문서에 걸치는 질문은 억지로 만들지 말고, 서로 이어진 항목에서 자연스럽게 찾는다.
- 근거 문장은 눈으로 옮겨 적지 말고 본문에서 복사한다.

세부구현:
1. 전체 본문을 한 번 출력해 읽는다. 표가 잘려 보이면 한 행씩 찍어 본다.
2. 질문마다 (질문, 정답 id 목록, 근거 문장 목록) 세 쪽짜리 자료를 목록으로 모은다.
3. 그 목록을 돌면서 id 목록은 파이프로, 근거 목록은 공백 파이프 파이프 공백으로 이어 붙인다.
   3-1. 이어 붙이는 순서가 서로 어긋나지 않게 같은 반복 안에서 처리한다.
4. 만든 행들을 DataFrame 으로 만들고 열 순서를 요구사항대로 맞춘다.
```

</details>

> 아래 셀을 먼저 실행해 **문서를 다 읽고** 시작하세요. 읽지 않고 쓴 질문은 5번 점검에서 걸립니다.

In [ ]:
# 24건을 전부 읽습니다 -- 질문은 이 안에서 나옵니다.
#  1번에서 만든 faq 를 그대로 쓰되, 아직 없으면 여기서 읽습니다(이 셀만 따로 실행해도 되게).
if 'faq' not in dir():
    import pandas as pd

    faq = pd.read_csv('../../day20_ReAct_멀티툴_에이전트/data/hr_faq.csv')

for row in faq.itertuples():
    print(f'[{row.id}] ({row.분류}) {row.제목}')
    print('   ', row.본문)
    print()

In [ ]:
# 질문마다 (질문, 정답 문서 id 목록, 근거 문장 목록) 을 모은다.
#  q1 은 일부러 넓게 잡은 질문이다 -- 6번에서 이 문항을 고친다.
rows = [
    ('휴가는 어떤 것들이 있고 며칠씩 쓸 수 있나요?',
     ['h01', 'h04', 'h06', 'h08', 'h09'],
     ['1년을 채우면 15일이 한꺼번에 생깁니다',
      '본인 결혼은 5일, 자녀 결혼은 1일의 경조휴가가 주어집니다',
      '아파서 일하기 어려우면 연간 10일까지 병가를 쓸 수 있습니다',
      '출산하는 직원은 출산 전후로 90일의 휴가를 받습니다',
      '배우자가 출산하면 20일의 휴가를 쓸 수 있습니다']),
    ('남은 연차를 안 쓰면 수당으로 받을 수 있나요?',
     ['h01', 'h03'],
     ['남은 연차는 다음 해로 이월되지 않습니다',
      '촉진 안내를 받고도 쓰지 않은 연차는 수당으로 보상되지 않습니다']),
    ('결혼하면 휴가와 지원금을 어떻게 받나요?',
     ['h04', 'h05'],
     ['본인 결혼은 5일, 자녀 결혼은 1일의 경조휴가가 주어집니다',
      '결혼과 출산, 상을 당했을 때 회사가 정한 금액의 경조사비를 지급합니다']),
    ('재택하는 날에도 출퇴근 기록을 남겨야 하나요?',
     ['h10', 'h12'],
     ['재택하는 날에도 정해진 근무시간에는 연락이 닿아야 합니다',
      '재택근무를 한 날도 같은 방법으로 기록합니다']),
    ('급여일이 휴일이면 언제 들어오나요?',
     ['h15'],
     ['급여는 매월 25일에 지급되고 그날이 휴일이면 앞당겨 지급합니다']),
    ('외부 교육 수강료를 지원받을 수 있나요?',
     ['h21'],
     ['직무와 관련한 외부 교육을 들으면 수강료의 절반을 연간 100만원까지 지원합니다']),
    ('건강검진 비용은 누가 부담하나요?',
     ['h19'],
     ['검진 비용은 회사가 지정한 병원에서 받으면 전액 지원됩니다']),
    ('초과근무를 미리 신청하지 않으면 어떻게 되나요?',
     ['h13'],
     ['승인 없이 남은 시간은 수당으로 인정되지 않습니다']),
]

# id 와 근거를 같은 반복 안에서 이어 붙인다 -- 따로 만들면 순서가 어긋나도 에러가 나지 않는다
records = []
for i, (query, gold, evidence) in enumerate(rows, 1):
    records.append({'query_id': f'q{i}',
                    'query': query,
                    'gold_ids': '|'.join(gold),
                    'evidence': ' || '.join(evidence)})

my_eval = pd.DataFrame(records, columns=['query_id', 'query', 'gold_ids', 'evidence'])
print(f'{len(my_eval)}문항')
display(my_eval[['query_id', 'query', 'gold_ids']])

In [ ]:
# [자가채점]
assert list(my_eval.columns) == ['query_id', 'query', 'gold_ids', 'evidence'], '열 이름과 순서를 맞춰 주세요'
assert len(my_eval) >= 8, '질문을 8개 이상 만드세요'
assert my_eval['query'].duplicated().sum() == 0, '같은 질문이 두 번 들어 있습니다'

gold_lists = [g.split('|') for g in my_eval['gold_ids']]
evidence_lists = [e.split(' || ') for e in my_eval['evidence']]
assert all(len(g) == len(e) for g, e in zip(gold_lists, evidence_lists)), \
    '정답 id 개수와 근거 문장 개수가 다른 문항이 있습니다'
assert sum(1 for g in gold_lists if len(g) >= 2) >= 2, '정답이 2개 이상인 문항이 2개 이상 필요합니다'
assert sum(1 for g in gold_lists if len(g) >= 4) == 1, "정답이 4개 이상인 '넓은 질문' 이 정확히 1개 필요합니다"
print('✅ 통과!')

**해설**: 여기서 한 일은 **판단**입니다. 코드는 마지막에 이어 붙이는 몇 줄뿐이고, 대부분의 시간은 문서를 읽고 "이 질문의 답이 정말 이 문서에 있나" 를 정하는 데 씁니다. 실무에서도 같습니다 — 평가셋 만들기의 어려움은 코드가 아니라 **기준을 정하고 끝까지 같은 기준으로 판단하는 것**입니다.

**모범답안이 고른 겹침**: 연차(`h01`)와 사용 촉진(`h03`), 경조휴가(`h04`)와 경조사비(`h05`), 재택(`h10`)과 근태 기록(`h12`). 억지로 두 문서를 묶은 것이 아니라 **한 질문의 답이 실제로 두 곳에 나뉘어 있는** 자리를 고른 것입니다. 이런 문항이 있어야 Recall 이 Hit 과 다른 것을 말합니다.

**`q1` 은 일부러 넓게** 잡았습니다(정답 5개). 6번에서 이 문항 하나가 표 전체를 어떻게 끌어내리는지 보고, 그것을 고치는 것이 마지막 단계입니다.

**흔한 실수 두 가지**

1. **근거를 요약해서 적기** — "연차는 이월 안 됨" 처럼 자기 말로 줄이면 원문과 달라져 5번에서 전부 걸립니다. 근거는 **복사**하는 것입니다.
2. **id 목록과 근거 목록을 따로 만들기** — 순서가 어긋나도 개수만 같으면 **에러가 나지 않습니다.** `h04` 의 근거 자리에 `h05` 의 문장이 들어가 있어도 5번 검증까지 조용히 통과할 수 있으므로, 모범답안처럼 **같은 반복 안에서** 짝지어 이어 붙입니다.

## 5. 내보내기 전 점검 — 이 평가셋을 믿어도 되나

**배경**: 평가셋에 결함이 있으면 **지표가 조용히 거짓말을 합니다.** 숫자는 멀쩡하게 나오는데 그 숫자가 아무것도 말해 주지 않습니다. 그래서 쓰기 전에 기계로 한 번 훑습니다.

| 점검 | 통과 못 하면 |
|---|---|
| 1. 정답 문서 id 가 코퍼스에 실제로 있는가 | 그 문항은 영원히 0점이 되는데 **에러는 안 난다** |
| 2. 근거 문장이 그 문서 원문에 있는가 | 라벨의 근거가 지어낸 것이다 |
| 3. 질문이 원문을 베끼지 않았는가 | 점수가 실제보다 높게 나온다 |
| 4. 한 문서가 여러 문항의 정답으로 쓰이는가 | 결함은 아니지만, 그 문서만 잘 찾아도 점수가 오른다 |

**요구사항**: 아래 제공 셀을 실행한 뒤 네 가지를 확인하세요.

- **`bad_ids`** — 코퍼스에 없는 정답 id 의 리스트. 비어 있어야 정상입니다.
- **`bad_evidence`** — 근거가 원문에 없는 것의 리스트. 원소는 **`(query_id, doc_id)` 튜플**입니다. 비어 있어야 정상입니다.
- **`check`** — 문항별 점검 표 DataFrame. 열 이름과 순서는 **`query_id`, `겹침`** 이고, `겹침` 은 **그 문항의 정답 문서들과의 겹침 평균**입니다. 만든 뒤 `0.6` 을 넘는 문항이 몇 개인지 출력하세요. (원본 `my_eval` 에 열을 붙이지 말고 **따로** 만드세요 — 뒤 문제들이 `my_eval` 을 그대로 씁니다.)
- **`shared`** — 두 번 이상 정답으로 쓰인 문서 id 를 모은 리스트(정렬). 결함이 아니므로 출력만 하면 됩니다.

**예시**

```
없는 문서 id  : []
원문에 없는 근거: []
겹침이 0.6 을 넘는 문항: 0개
두 번 이상 쓰인 정답 문서: ['h01', 'h04']
```

<details><summary>힌트</summary>

```text
접근방법:
- 문항을 한 번 돌면서 정답 id 와 근거 문장을 짝지어 검사하면 1번과 2번이 함께 끝난다.
- 겹침은 한 문항의 정답 문서마다 재서 평균을 낸다.

세부구현:
1. 코퍼스의 id 를 집합으로 한 번 모아 둔다. 문항마다 표를 뒤지면 느리다.
2. 문항을 돌며 정답 id 와 근거 문장을 짝지어 반복한다.
   2-1. id 가 집합에 없으면 그 id 를 첫 목록에 넣는다.
   2-2. id 는 있는데 근거가 원문에 없으면 문항 번호와 id 를 짝으로 두 번째 목록에 넣는다.
3. 겹침은 문항의 정답 문서마다 재서 평균을 내고 새 열로 붙인다.
4. 모든 정답 id 를 한 목록에 펼친 뒤 두 번 이상 나오는 것만 중복 없이 모은다.
```

</details>

In [ ]:
# [제공 코드] 질문이 원문을 얼마나 베꼈는지 재는 함수 -- 이 셀은 실행만 하세요.
#  질문을 세 글자씩 잘라 그 덩어리가 문서 본문에 그대로 나오는 비율을 봅니다.
#  두 글자로 세면 한국어 조사·어미가 자주 겹쳐 베끼지 않은 질문까지 걸립니다.
import re


def overlap_ratio(query, text):
    # 띄어쓰기만 다른 표현도 같은 것으로 세도록 공백을 모두 지운다
    squeezed = re.sub(r'\s+', '', query)
    # 세 글자씩 한 칸씩 밀며 잘라 중복 없이 모은다
    grams = {squeezed[i:i + 3] for i in range(len(squeezed) - 2)}
    body_text = re.sub(r'\s+', '', text)
    return sum(1 for g in grams if g in body_text) / len(grams)


print('겹침 측정 함수 준비 완료')

In [ ]:
# 코퍼스에 있는 id 를 집합으로 한 번에 모아 둔다(문항마다 표를 뒤지면 느리다)
faq_ids = set(faq['id'])

bad_ids, bad_evidence = [], []
for row in my_eval.itertuples():
    ids = row.gold_ids.split('|')
    sentences = row.evidence.split(' || ')
    for doc_id, sentence in zip(ids, sentences):
        if doc_id not in faq_ids:
            bad_ids.append(doc_id)
        elif not quoted_in_faq(sentence, doc_id):
            # id 가 있을 때만 검사한다 -- 없는 id 로 본문을 꺼내면 KeyError 가 난다
            bad_evidence.append((row.query_id, doc_id))

print('없는 문서 id  :', bad_ids)
print('원문에 없는 근거:', bad_evidence)

# 질문이 정답 문서를 얼마나 베꼈는지 -- 문항의 정답 문서마다 재서 평균을 낸다.
#  my_eval 에 열을 붙이지 않고 따로 표를 만든다(뒤 문제들이 my_eval 을 그대로 쓴다)
check_rows = []
for row in my_eval.itertuples():
    ids = row.gold_ids.split('|')
    check_rows.append({'query_id': row.query_id,
                       '겹침': sum(overlap_ratio(row.query, body[d]) for d in ids) / len(ids)})

check = pd.DataFrame(check_rows, columns=['query_id', '겹침'])
print(f"겹침이 0.6 을 넘는 문항: {(check['겹침'] > 0.6).sum()}개")

# 한 문서가 여러 문항의 정답인 경우 -- 결함은 아니지만 알고는 있어야 한다
gold_all = [d for row in my_eval.itertuples() for d in row.gold_ids.split('|')]
shared = sorted({d for d in gold_all if gold_all.count(d) > 1})
print('두 번 이상 쓰인 정답 문서:', shared)

In [ ]:
# [자가채점]
assert bad_ids == [], f'코퍼스에 없는 문서 id 입니다: {bad_ids}'
assert bad_evidence == [], f'근거 문장이 그 문서 원문에 없습니다: {bad_evidence}'
assert list(check.columns) == ['query_id', '겹침'], 'check 의 열 이름과 순서를 맞춰 주세요'
assert len(check) == len(my_eval), '모든 문항이 점검 표에 들어가야 합니다'
assert (check['겹침'] > 0.6).sum() == 0, '원문을 그대로 베낀 질문이 있습니다 - 물어볼 법한 말로 고쳐 주세요'
assert isinstance(shared, list), 'shared 는 문서 id 리스트입니다'
print('✅ 통과!')

**해설**: 네 점검 중 **1번이 가장 무섭습니다.** 없는 id 를 정답이라고 적어 두면 그 문항은 무엇을 해도 0점인데, **에러가 나지 않습니다.** 검색은 정상으로 돌고 점수만 낮게 나오니 "검색기가 나쁘다" 고 잘못 결론 내리게 됩니다. 이 코퍼스는 청킹이 없어 id 가 흔들릴 일이 없지만, 문서를 잘라 쓰는 경우에는 **자르는 규칙을 바꾼 순간** 라벨 전체가 이렇게 됩니다.

**`elif` 인 이유**: id 가 없는데 `body[doc_id]` 로 본문을 꺼내면 `KeyError` 가 납니다. 점검 도구가 점검 중에 죽으면 나머지 문항은 보지도 못합니다.

**3번 겹침 점검이 잡는 것**: "검진 비용은 회사가 지정한 병원에서 받으면 전액 지원되나요?" 처럼 원문을 거의 그대로 질문으로 만들면 검색이 못 맞힐 수가 없습니다. 그런 평가셋으로 재면 점수는 높게 나오는데, **실제 사용자는 그렇게 묻지 않습니다.** 모범답안 8문항의 겹침은 최대 0.385 로 문턱 아래에 있습니다.

**4번은 결함이 아닙니다**: 모범답안에서는 `h01`(연차)과 `h04`(경조휴가)가 두 문항의 정답으로 쓰입니다. 다만 한 문서가 여러 문항의 답이면 **그 문서 하나만 잘 찾아도 점수가 여러 문항에서 함께 오르므로**, 점수를 읽을 때 감안해야 합니다.

## 6. 내 평가셋으로 검색기 재기 — 그리고 평가셋을 고치기

**배경**: 평가셋이 준비됐으니 이제 **재 봅니다.** 그런데 이 과제의 진짜 목적은 점수를 내는 것이 아닙니다. 점수를 읽고 **어디가 문제인지 찾아 고치는 것**입니다. 낮은 점수가 검색기 탓인지 평가셋 탓인지 가릴 줄 알아야 그다음에 무엇을 고칠지 정할 수 있습니다.

세 단계로 갑니다. **1단계** K 별로 재서 표를 만들고, **2단계** 문항별로 쪼개 보아 어느 문항이 표를 끌어내리는지 찾고, **3단계** 그 문항을 고쳐 다시 잽니다.

> 아래 제공 셀의 지표 네 개는 **RAG 평가 과제에서 여러분이 직접 만든 것과 같은 함수**입니다. 여기서는 만드는 것이 아니라 **쓰는** 것이 목적이라 그대로 드립니다.

In [ ]:
# [제공 코드] 검색 평가 지표 네 개 — 이 셀은 실행만 하세요.
#  RAG 평가 과제에서 여러분이 직접 만든 것과 같은 함수입니다. 여기서는 도구로 씁니다.
#  네 함수의 인자는 모두 같습니다: ranked(검색 결과 id 를 1위부터), gold(정답 id 목록), k.
def hit_at_k(ranked, gold, k):
    # 상위 k개 안에 정답이 하나라도 있으면 1.0 -- 맞혔나 못 맞혔나만 본다
    return 1.0 if any(r in gold for r in ranked[:k]) else 0.0


def precision_at_k(ranked, gold, k):
    # 꺼내 온 k개 중 몇 개가 정답이었나 -- 나누는 수는 결과 길이가 아니라 언제나 k
    return sum(1 for r in ranked[:k] if r in gold) / k


def recall_at_k(ranked, gold, k):
    # 정답 전체 중 몇 개를 건졌나 -- 나누는 수가 정답 개수라 정답이 많으면 낮아진다
    return sum(1 for r in ranked[:k] if r in gold) / len(gold)


def mrr_at_k(ranked, gold, k):
    # 첫 정답이 몇 위였나 -- 1위면 1, 2위면 0.5. 순위를 보는 유일한 지표다
    for rank, doc_id in enumerate(ranked[:k], 1):
        if doc_id in gold:
            return 1 / rank
    return 0.0


print('지표 네 개 준비 완료')

### 1단계 — K 를 바꿔 가며 표 만들기

**요구사항**: `K = 1, 3, 5, 10` 각각에 대해 여러분의 평가셋 전체를 네 지표로 재어 DataFrame **`k_table`** 을 만드세요.

- 열 이름과 순서는 **`K`, `Hit`, `Precision`, `Recall`, `MRR`** 이고, 행은 K 가 작은 것부터입니다.
- 각 값은 **전체 문항의 평균**이고 **소수 셋째 자리까지 반올림**합니다.
- 문항마다 검색은 **한 번만** 하세요. 지표마다 다시 검색하면 느리고, 재는 대상도 흔들립니다.
- 만든 표를 `display()` 로 보세요.

**예시** (모범답안 8문항 기준 — 여러분의 값은 다릅니다)

```
   K    Hit  Precision  Recall    MRR
   1  0.875      0.875   0.688  0.875
   3  1.000      0.458   0.863  0.938
```

<details><summary>힌트</summary>

```text
접근방법:
- 바깥 반복은 K, 안쪽 반복은 문항이다.
- 문항마다 검색 결과를 한 번 받아 네 지표에 모두 넘긴다.

세부구현:
1. K 값들을 담은 목록을 만들고 결과를 모을 빈 목록을 준비한다.
2. K 마다 문항을 돌며 검색 결과와 정답 목록을 얻는다.
   2-1. 정답 목록은 파이프로 나눈 것이다.
   2-2. 네 지표를 각각 재어 한 문항의 값 네 개를 모은다.
3. 문항들의 값을 지표별로 평균 내고 반올림해 한 행으로 만든다.
4. 행들을 DataFrame 으로 만든다.
```

</details>

In [ ]:
rows = []
for k in [1, 3, 5, 10]:
    scores = []
    for row in my_eval.itertuples():
        gold = row.gold_ids.split('|')
        # 검색은 문항당 한 번만 -- 지표마다 다시 부르면 느리고 재는 대상도 흔들린다
        ranked = search_ids(row.query, k)
        scores.append((hit_at_k(ranked, gold, k),
                       precision_at_k(ranked, gold, k),
                       recall_at_k(ranked, gold, k),
                       mrr_at_k(ranked, gold, k)))
    # zip(*scores) 로 지표별로 묶어 평균을 낸다
    hit, precision, recall, mrr = [sum(v) / len(v) for v in zip(*scores)]
    rows.append({'K': k, 'Hit': round(hit, 3), 'Precision': round(precision, 3),
                 'Recall': round(recall, 3), 'MRR': round(mrr, 3)})

k_table = pd.DataFrame(rows, columns=['K', 'Hit', 'Precision', 'Recall', 'MRR'])
display(k_table)

In [ ]:
# [자가채점]
assert list(k_table.columns) == ['K', 'Hit', 'Precision', 'Recall', 'MRR'], '열 이름과 순서를 맞춰 주세요'
assert list(k_table['K']) == [1, 3, 5, 10], 'K 는 1, 3, 5, 10 순서입니다'
for name in ['Hit', 'Precision', 'Recall', 'MRR']:
    assert k_table[name].between(0, 1).all(), f'{name} 은 0 과 1 사이여야 합니다'

# K 를 키우면 더 넓게 보는 것이므로 Hit·Recall·MRR 은 절대 줄어들 수 없다.
#  줄었다면 상위 k개를 자르지 않았거나 정답 목록을 잘못 만든 것이다.
for name in ['Hit', 'Recall', 'MRR']:
    values = list(k_table[name])
    assert all(a <= b for a, b in zip(values, values[1:])), \
        f'{name} 이 K 가 커지는데 줄었습니다 - 상위 k개만 보고 있는지 확인하세요'
print('✅ 통과!')

**해설**: `Precision` 만 K 가 커질수록 떨어집니다(0.875 -> 0.188). 나누는 수가 `k` 라서 넓게 볼수록 정답이 아닌 것이 함께 딸려 오기 때문입니다. 나머지 셋은 **줄어들 수가 없습니다** — 더 넓게 봤는데 찾았던 것을 잃을 수는 없으니까요. 자가채점이 이 성질을 보는 이유입니다. 학생마다 값은 달라도 **이 관계만은 모두에게 같아야** 합니다.

**흔한 실수**: `ranked[:k]` 를 빠뜨리고 검색 결과 전체를 지표에 넘기면, K=1 인데도 뒤 순위의 정답을 세어 Hit@1 이 이상하게 높게 나옵니다. `search_ids` 가 이미 k개만 주므로 표는 그럴듯해 보이지만, K=10 으로 잰 값이 K=1 자리에 앉게 됩니다.

### 2단계 — 어느 문항이 표를 끌어내리는가

**배경**: 평균은 원인을 감춥니다. `Recall` 이 낮게 나왔다면 검색기가 못 찾은 것일 수도 있고, **질문이 너무 넓어 애초에 다 찾을 수 없는 것**일 수도 있습니다. 문항별로 쪼개 보면 갈립니다.

**요구사항**: `K=3` 으로 문항마다 재어 DataFrame **`per_item`** 을 만드세요.

- 열 이름과 순서는 **`query_id`, `정답수`, `Hit3`, `Recall3`** 입니다.
- `정답수` 는 그 문항의 정답 문서 개수, `Hit3`·`Recall3` 은 K=3 으로 잰 값(소수 셋째 자리 반올림)입니다.
- `Recall3` 이 낮은 것부터 보이도록 **`Recall3` 오름차순으로 정렬**해 `display()` 하세요.

그리고 아래 서술형 답안 셀에 답하세요.

**질문**: 정답이 4개 이상인 그 '넓은 질문'의 `Recall3` 을 보세요. 이 값이 **낮을 수밖에 없는 이유**는 무엇인가요? 검색기를 아무리 잘 고쳐도 이 문항의 `Recall3` 이 넘을 수 없는 한계값이 있습니다. 그 값은 얼마이고 왜 그런가요?

<details><summary>힌트</summary>

```text
접근방법:
- 1단계와 같은 계산인데 평균을 내지 않고 문항별로 남긴다.

세부구현:
1. 문항을 돌며 정답 목록과 검색 결과를 얻는다.
2. 문항 번호, 정답 개수, 두 지표 값을 한 행으로 모은다.
3. DataFrame 으로 만든 뒤 Recall 열 기준으로 오름차순 정렬한다.
```

</details>

In [ ]:
rows = []
for row in my_eval.itertuples():
    gold = row.gold_ids.split('|')
    ranked = search_ids(row.query, 3)
    rows.append({'query_id': row.query_id,
                 '정답수': len(gold),
                 'Hit3': round(hit_at_k(ranked, gold, 3), 3),
                 'Recall3': round(recall_at_k(ranked, gold, 3), 3)})

per_item = pd.DataFrame(rows, columns=['query_id', '정답수', 'Hit3', 'Recall3'])
display(per_item.sort_values('Recall3'))

In [ ]:
# [자가채점]
assert list(per_item.columns) == ['query_id', '정답수', 'Hit3', 'Recall3'], '열 이름과 순서를 맞춰 주세요'
assert len(per_item) == len(my_eval), '모든 문항이 들어가야 합니다'

# 상위 3개만 보는데 정답이 5개라면 Recall 은 아무리 잘해도 3/5 를 넘을 수 없다.
#  이 관계는 어떤 평가셋에서도 성립하므로, 깨졌다면 계산이 틀린 것이다.
for row in per_item.itertuples():
    assert row.Recall3 <= min(1.0, 3 / row.정답수) + 1e-9, \
        f'{row.query_id}: 상위 3개로는 나올 수 없는 Recall 입니다 - 정답 목록을 확인하세요'
print('✅ 통과!')

**서술형 답안**

상위 **3개**만 꺼내 보는데 정답이 **5개**이므로, 세 개를 전부 맞혀도 `Recall3` 은 **3/5 = 0.6** 이 최대입니다. 모범답안의 `q1` 은 0.400 이 나왔지만, 검색기를 완벽하게 고쳐도 0.6 을 넘을 수 없습니다. **검색기의 문제가 아니라 질문의 문제**입니다 — 답이 다섯 문서에 흩어진 질문을 K=3 으로 재고 있으니까요.

그래서 `Recall` 은 **정답 개수와 함께** 읽어야 합니다. 낮은 `Recall` 을 보고 검색기를 고치기 시작하면 고칠 수 없는 것을 붙들게 됩니다. 할 수 있는 일은 둘입니다 — **질문을 쪼개거나**, 그 문항만 따로 **더 큰 K 로** 재거나.

**해설**: 이 단계가 이 과제의 핵심입니다. 학생이 자주 하는 오해가 "점수가 낮다 = 검색기가 나쁘다" 인데, **평가셋이 만든 한계**인 경우가 실무에서 매우 흔합니다. `Recall@K <= K / 정답수` 라는 부등식은 검색기와 무관하게 **산수로 정해집니다.**

자가채점이 이 부등식을 검사하는 것도 같은 이유입니다. 학생마다 평가셋이 다르니 값은 고정할 수 없지만, **이 관계만은 누구에게나 참**이라 옳게 계산했으면 반드시 통과하고 잘못 계산하면 깨집니다.

**모범답안 문항별 값(K=3)**: `q1` 정답 5개 Recall 0.400 / `q2` 정답 2개 0.500 / `q3`·`q4` 정답 2개 1.000 / 나머지 단일 정답 문항 1.000. 정답이 여럿인 문항에서만 Hit 과 Recall 이 갈리는 것을 표에서 바로 볼 수 있습니다.

### 3단계 — 질문을 고쳐 다시 재기

**배경**: 2단계에서 찾은 그 넓은 질문은 사실 **여러 질문을 하나로 묶은 것**입니다. 쪼개면 각 문항의 정답이 줄고, 그러면 `Recall` 이 다시 **검색기의 성능을 재는 숫자**가 됩니다.

**요구사항**:
- 그 넓은 질문을 **주제가 다른 두 질문으로 쪼개** 새 평가셋 **`my_eval2`** 를 만드세요. 원래 문항은 **빼고** 쪼갠 둘을 **넣습니다**(나머지 문항은 그대로).
  - 열 이름과 순서는 `my_eval` 과 같은 **`query_id`, `query`, `gold_ids`, `evidence`** 입니다.
  - 쪼갠 두 문항의 `query_id` 는 기존과 겹치지 않게 붙이세요(예: `q1a`, `q1b`).
  - 근거 문장은 여기서도 **원문 그대로**입니다.
- `my_eval2` 로 **`k_table2`** 를 만드세요. 만드는 방법은 1단계와 같습니다(열도 같습니다).
- 두 표의 **K=3 행 `Recall`** 을 나란히 출력해 얼마나 달라졌는지 보세요.

그리고 아래 서술형 답안 셀에 답하세요.

**질문**: `Recall` 이 올라갔습니다. 그렇다면 **검색기가 좋아진 것인가요?** 아니라면 이 작업으로 실제로 좋아진 것은 무엇인가요?

<details><summary>힌트</summary>

```text
접근방법:
- 평가셋을 다시 만드는 것이지 검색을 바꾸는 것이 아니다.
- 표 만드는 코드는 1단계 것을 그대로 쓰되 대상만 바꾼다.

세부구현:
1. 기존 문항에서 넓은 질문만 뺀 목록을 만든다.
2. 쪼갠 두 문항을 같은 열 구조로 만들어 이어 붙인다.
3. 1단계와 같은 방법으로 K 별 표를 만든다.
4. 두 표에서 K 가 3 인 행의 Recall 값을 꺼내 함께 출력한다.
```

</details>

In [ ]:
# 넓은 질문 q1 을 뺀 나머지를 그대로 두고, 쪼갠 두 문항을 새로 붙인다
kept = my_eval[my_eval['query_id'] != 'q1']

split_rows = [
    {'query_id': 'q1a',
     'query': '연차는 1년에 며칠 생기나요?',
     'gold_ids': 'h01',
     'evidence': '1년을 채우면 15일이 한꺼번에 생깁니다'},
    {'query_id': 'q1b',
     'query': '출산과 관련한 휴가는 며칠씩인가요?',
     'gold_ids': 'h08|h09',
     'evidence': '출산하는 직원은 출산 전후로 90일의 휴가를 받습니다'
                 ' || 배우자가 출산하면 20일의 휴가를 쓸 수 있습니다'},
]

my_eval2 = pd.concat([kept, pd.DataFrame(split_rows)], ignore_index=True)


def make_k_table(evalset):
    """평가셋 하나를 K 별로 재어 표로 돌려준다(1단계와 같은 계산)."""
    rows = []
    for k in [1, 3, 5, 10]:
        scores = []
        for row in evalset.itertuples():
            gold = row.gold_ids.split('|')
            ranked = search_ids(row.query, k)
            scores.append((hit_at_k(ranked, gold, k),
                           precision_at_k(ranked, gold, k),
                           recall_at_k(ranked, gold, k),
                           mrr_at_k(ranked, gold, k)))
        hit, precision, recall, mrr = [sum(v) / len(v) for v in zip(*scores)]
        rows.append({'K': k, 'Hit': round(hit, 3), 'Precision': round(precision, 3),
                     'Recall': round(recall, 3), 'MRR': round(mrr, 3)})
    return pd.DataFrame(rows, columns=['K', 'Hit', 'Precision', 'Recall', 'MRR'])


k_table2 = make_k_table(my_eval2)
display(k_table2)

# K=3 행만 꺼내 나란히 본다 -- 검색기는 그대로이고 평가셋만 바뀌었다
before = k_table.loc[k_table['K'] == 3, 'Recall'].iloc[0]
after = k_table2.loc[k_table2['K'] == 3, 'Recall'].iloc[0]
print(f'문항 {len(my_eval)}개 -> {len(my_eval2)}개')
print(f'Recall@3 : {before} -> {after}')

In [ ]:
# [자가채점]
assert list(my_eval2.columns) == ['query_id', 'query', 'gold_ids', 'evidence'], '열 이름과 순서를 맞춰 주세요'
assert len(my_eval2) == len(my_eval) + 1, '넓은 질문 1개를 빼고 2개를 넣었으므로 문항이 하나 늘어야 합니다'
assert my_eval2['query_id'].is_unique, 'query_id 가 겹칩니다'

gold2 = [g.split('|') for g in my_eval2['gold_ids']]
assert max(len(g) for g in gold2) < 4, '정답이 4개 이상인 문항이 아직 남아 있습니다'
assert all(len(g) == len(e.split(' || ')) for g, e in zip(gold2, my_eval2['evidence'])), \
    '쪼갠 문항의 정답 id 개수와 근거 문장 개수가 다릅니다'
# 새로 적은 근거도 원문에 있어야 한다 -- 쪼개면서 요약해 적기 쉬운 자리다
for row in my_eval2.itertuples():
    for doc_id, sentence in zip(row.gold_ids.split('|'), row.evidence.split(' || ')):
        assert quoted_in_faq(sentence, doc_id), f'{row.query_id} 의 근거가 {doc_id} 원문에 없습니다'

assert list(k_table2.columns) == list(k_table.columns), 'k_table2 는 k_table 과 같은 열 구조입니다'
for name in ['Hit', 'Recall', 'MRR']:
    values = list(k_table2[name])
    assert all(a <= b for a, b in zip(values, values[1:])), f'{name} 이 K 가 커지는데 줄었습니다'
print('✅ 통과!')

**서술형 답안**

**아닙니다. 검색기는 한 글자도 바뀌지 않았습니다.** 같은 색인, 같은 임베딩, 같은 `search_ids` 로 쟀습니다. 올라간 것은 검색 성능이 아니라 **평가셋이 검색기를 재는 정확도**입니다.

고쳐진 것은 이것입니다. 전에는 `Recall` 이 낮게 나와도 그것이 **검색기가 못 찾아서인지, 질문이 넓어서인지** 구분할 수 없었습니다. 이제는 어느 문항이든 정답이 셋 이하라 K=3 으로 다 찾을 수 있고, 그래서 낮은 `Recall` 이 나오면 **그건 검색기 탓**입니다. 지표가 비로소 **고칠 수 있는 것**을 가리키게 된 것입니다.

반대로 조심할 것도 있습니다. 질문을 쪼개면 점수는 올라가므로, **점수를 올리려고 평가셋을 손보는 것**은 스스로를 속이는 일이 됩니다. 쪼갠 이유가 "점수가 낮아서" 가 아니라 "한 질문에 여러 질문이 섞여 있어서" 여야 합니다.

**해설**: 모범답안 기준으로 문항이 8개에서 9개로 늘고, 문항당 평균 정답 개수가 1.88 에서 1.44 로 줄면서 `Recall@3` 이 **0.863 에서 0.944** 로 올랐습니다. `Precision@3` 은 0.458 에서 0.444 로 거의 그대로입니다 — 검색기가 그대로이니 당연합니다.

**쪼갤 때의 기준**: 그냥 반으로 자르는 것이 아니라 **주제가 다른 두 질문**으로 나눕니다. 모범답안은 "휴가는 어떤 것들이 있고 며칠씩" 을 연차(`h01`)와 출산 관련 휴가(`h08`·`h09`)로 갈랐습니다. 이렇게 나누면 각 문항의 답이 한두 문서에 모입니다.

**여기서 버린 것도 있습니다**: 원래 `q1` 의 정답이던 경조휴가(`h04`)와 병가(`h06`)는 쪼갠 두 문항 어디에도 들어가지 않았습니다. 그 주제를 재고 싶다면 **문항을 더 만들어야** 합니다. 질문을 쪼개는 일은 공짜가 아니라 **평가셋이 커지는 일**입니다.

**대안**: 쪼갤 수 없는 질문("점검 항목을 전부 알려 줘" 같은 것)이라면, 그 문항만 따로 묶어 **더 큰 K 로** 재는 편이 정직합니다. K=3 으로 재면서 낮은 `Recall` 을 검색기 탓으로 적는 것보다 낫습니다.

---
수고했어요! 이 과제에서 여러분은 **재는 도구를 직접 만들었습니다.**

| 한 일 | 남길 것 |
|---|---|
| 한 행이 한 문서인 코퍼스를 색인 | 자를 것이 없으면 라벨이 문서 id 그대로다 |
| 질문을 쓰고 정답 라벨과 근거 문장을 붙임 | 근거 문장이 있어야 나중에 그 라벨을 검수할 수 있다 |
| 내보내기 전에 기계로 점검 | 없는 id·지어낸 근거·베낀 질문은 **에러 없이** 지표를 망친다 |
| 문항별로 쪼개 원인을 찾음 | `Recall` 은 **정답 개수와 함께** 읽는다 |
| 질문을 고쳐 다시 잼 | 점수가 오른 것과 **검색기가 좋아진 것**은 다른 이야기다 |

한 가지만 남긴다면: **평가셋이 틀리면 그것으로 잰 점수가 전부 함께 틀립니다.** 그런데 화면에는 멀쩡해 보이는 숫자가 찍히기 때문에 알아채기가 어렵습니다. 그래서 재기 전에 재는 도구부터 검수합니다.

> 라벨을 **모델에게 시키고** 그 판정을 검증하는 방법이 궁금하다면 `교안_03_평가셋_구축.ipynb` 를 보세요. 이 과제에서 손으로 한 일을 자동화하는 절차입니다.